In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
from io import StringIO
import os
from datetime import datetime, timedelta
import logging

In [9]:
import warnings

# Ignore all warnings
warnings.filterwarnings('ignore')

In [3]:
# Load CSV file
stock_data = pd.read_csv('stock_data.csv')

# Show the first few rows
print(stock_data.head())

# Show all column names
print(stock_data.columns.tolist())

         Date    A.close     A.open  AAPL.close  AAPL.open  ABBV.close  \
0  22/03/2010  21.563597  21.269431    6.763536   6.634736         NaN   
1  23/03/2010  21.813002  21.601970    6.872173   6.790318         NaN   
2  24/03/2010  21.678694  21.729854    6.902567   6.850505         NaN   
3  25/03/2010  22.094370  21.806600    6.820715   6.949215         NaN   
4  26/03/2010  21.940891  22.158317    6.948610   6.889927         NaN   

   ABBV.open  ABNB.close  ABNB.open  ABT.close  ...  XYL.close  XYL.open  \
0        NaN         NaN        NaN  18.598389  ...        NaN       NaN   
1        NaN         NaN        NaN  18.760651  ...        NaN       NaN   
2        NaN         NaN        NaN  18.598389  ...        NaN       NaN   
3        NaN         NaN        NaN  18.477552  ...        NaN       NaN   
4        NaN         NaN        NaN  18.263506  ...        NaN       NaN   

   YUM.close   YUM.open  ZBH.close   ZBH.open  ZBRA.close  ZBRA.open  \
0  20.547918  20.354020  5

In [11]:
# COMPUTING DAILY RETURNS (CLOSE TO CLOSE) --- NOTES

# df[return_col] = df[col].pct_change()
#  calculates the percentage change between the current day’s close price and the previous day’s close price for each ticker.

# Formula used
# Return_t = (Close_t / Close_t-1) - 1

In [ ]:
##  Long format


# Load your wide format data
df = pd.read_csv('stock_data.csv')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

# Find close price columns
close_columns = [col for col in df.columns if col.endswith('.close')]

# Initialize an empty DataFrame to store the long-format data
long_df = pd.DataFrame()

# Loop through tickers to reshape and compute returns
for col in close_columns:
    ticker = col.replace('.close', '')
    
    temp_df = df[['Date', col]].copy()
    temp_df['Ticker'] = ticker
    temp_df['Return'] = temp_df[col].pct_change() 
    
    # Rename price column for clarity
    temp_df = temp_df.rename(columns={col: 'Close'})
    
    # Drop NaN returns (optional)
    temp_df = temp_df.dropna()

    # Append to long format dataframe
    long_df = pd.concat([long_df, temp_df], axis=0)

# Reorder columns
long_df = long_df[['Date', 'Ticker', 'Close', 'Return']]

# Sort by date/ticker
long_df = long_df.sort_values(['Date', 'Ticker'])

# Save to CSV
long_df.to_csv('long_format_stock_returns.csv', index=False)

print("✅ Long format file saved as long_format_stock_returns.csv")



In [10]:
# Long returns
long_returns = pd.read_csv('long_format_stock_returns.csv')
# Show the first few rows
print(long_returns.head())

# Show all column names
print(long_returns.columns.tolist())

         Date Ticker      Close    Return
0  2010-03-23      A  21.813002  0.011566
1  2010-03-23   AAPL   6.872173  0.016062
2  2010-03-23    ABT  18.760651  0.008725
3  2010-03-23   ACGL   7.951624  0.001330
4  2010-03-23    ACN  32.192348 -0.000707
['Date', 'Ticker', 'Close', 'Return']


In [ ]:
# WIDE FORMAT

df = pd.read_csv('stock_data.csv')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

# Create new return columns next to each close price
for col in df.columns:
    if col.endswith('.close'):
        ticker = col.replace('.close', '')
        return_col = ticker + '.return'
        
        df[return_col] = df[col].pct_change()

# Save to CSV
df.to_csv('wide_format_with_returns.csv', index=False)

print("✅ Wide format file saved as wide_format_with_returns.csv")


In [6]:
wide_returns = pd.read_csv('wide_format_with_returns.csv')
# Show the first few rows
print(wide_returns.head())

# Show all column names
print(wide_returns.columns.tolist())

         Date    A.close     A.open  AAPL.close  AAPL.open  ABBV.close  \
0  2010-03-22  21.563597  21.269431    6.763536   6.634736         NaN   
1  2010-03-23  21.813002  21.601970    6.872173   6.790318         NaN   
2  2010-03-24  21.678694  21.729854    6.902567   6.850505         NaN   
3  2010-03-25  22.094370  21.806600    6.820715   6.949215         NaN   
4  2010-03-26  21.940891  22.158317    6.948610   6.889927         NaN   

   ABBV.open  ABNB.close  ABNB.open  ABT.close  ...  WTW.return  WY.return  \
0        NaN         NaN        NaN  18.598389  ...         NaN        NaN   
1        NaN         NaN        NaN  18.760651  ...    0.009807   0.008007   
2        NaN         NaN        NaN  18.598389  ...   -0.012218  -0.007061   
3        NaN         NaN        NaN  18.477552  ...    0.011100  -0.018667   
4        NaN         NaN        NaN  18.263506  ...   -0.001255   0.003170   

   WYNN.return  XEL.return  XOM.return  XYL.return  YUM.return  ZBH.return  \
0       

In [ ]:
# RETURNS ONLY WIDE

# Load your original wide format data
df = pd.read_csv('stock_data.csv')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

# Compute returns for each ticker's close price
for col in df.columns:
    if col.endswith('.close'):
        ticker = col.replace('.close', '')
        return_col = ticker + '.return'
        
        df[return_col] = df[col].pct_change()

# Drop all 'open' and 'close' columns, keep returns + Date
columns_to_keep = ['Date'] + [col for col in df.columns if col.endswith('.return')]
returns_only_df = df[columns_to_keep]

# Save returns-only wide format CSV
returns_only_df.to_csv('returns_only_wide_format.csv', index=False)

print("✅ Returns-only wide format file saved as returns_only_wide_format.csv")


In [14]:
#returns_only_df = pd.read_csv('returns_only_wide_format.csv')
# Show the first few rows
print(returns_only_df.head())

# Show all column names
print(returns_only_df.columns.tolist())

        Date  A.return  AAPL.return  ABBV.return  ABNB.return  ABT.return  \
0 2010-03-22       NaN          NaN          NaN          NaN         NaN   
1 2010-03-23  0.011566     0.016062          NaN          NaN    0.008725   
2 2010-03-24 -0.006157     0.004423          NaN          NaN   -0.008649   
3 2010-03-25  0.019174    -0.011858          NaN          NaN   -0.006497   
4 2010-03-26 -0.006947     0.018751          NaN          NaN   -0.011584   

   ACGL.return  ACN.return  ADBE.return  ADI.return  ...  WTW.return  \
0          NaN         NaN          NaN         NaN  ...         NaN   
1     0.001330   -0.000707     0.007725    0.011972  ...    0.009807   
2     0.002658   -0.014387     0.036627   -0.045021  ...   -0.012218   
3    -0.002650   -0.006461    -0.017529   -0.004130  ...    0.011100   
4     0.000399    0.021195    -0.010036   -0.015549  ...   -0.001255   

   WY.return  WYNN.return  XEL.return  XOM.return  XYL.return  YUM.return  \
0        NaN          NaN  

In [12]:
# COMPUTING EXCESS  RETURNS
# Formula
# Excess Return = Asset Return - Risk-Free Rate

# The Fin-GAN used the U.S. 3-Month Treasury Bill rate as the risk-free rate.
# This is often retrieved from FRED (Federal Reserve Economic Data).

In [13]:
# Retrieving risk-free rate
import pandas_datareader.data as web

# Get 3-Month Treasury Bill Rate from FRED
rf = web.DataReader('TB3MS', 'fred', start='2010-01-01', end='2025-01-01')

# Convert monthly rate (%) to daily and decimal (e.g., 5% -> 0.05)
rf_daily = rf.resample('B').ffill() / 100  # 'B' = business days
rf_daily.rename(columns={'TB3MS': 'risk_free_rate'}, inplace=True)

# Preview
print(rf_daily.head())


            risk_free_rate
DATE                      
2010-01-01          0.0006
2010-01-04          0.0006
2010-01-05          0.0006
2010-01-06          0.0006
2010-01-07          0.0006


In [19]:
# merge into the return only df (wide format)

# Convert 'Date' to datetime if it's not already
returns_only_df['Date'] = pd.to_datetime(returns_only_df['Date'])

# Merge the risk-free rate dataframe with your returns
excess_returns_df = pd.merge(returns_only_df, rf_daily, left_on='Date', right_index=True, how='left')

# Subtract risk-free rate from each return column
for col in excess_returns_df.columns:
    if col.endswith('.return'):
        excess_returns_df[col] = excess_returns_df[col] - excess_returns_df['risk_free_rate']

# Drop the risk-free rate column if not needed
#excess_returns_df = excess_returns_df.drop(columns=['risk_free_rate'])

#Save to CSV
excess_returns_df.to_csv('excess_returns_only_wide.csv', index=False)

print("Excess returns wide format file saved as excess_returns_only_wide.csv")


Excess returns wide format file saved as excess_returns_only_wide.csv


In [20]:
excess_returns_df.head()

,Date,A.return,AAPL.return,ABBV.return,ABNB.return,ABT.return,ACGL.return,ACN.return,ADBE.return,ADI.return,...,WY.return,WYNN.return,XEL.return,XOM.return,XYL.return,YUM.return,ZBH.return,ZBRA.return,ZTS.return,risk_free_rate
0,2010-03-22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0015
1,2010-03-23,0.010066,0.014562,NaN,NaN,0.007225,-0.000170,-0.002207,0.006225,0.010472,...,0.006507,-0.004244,0.000617,-0.001799,NaN,0.001121,0.002968,-0.007474,NaN,0.0015
2,2010-03-24,-0.007657,0.002923,NaN,NaN,-0.010149,0.001158,-0.015887,0.035127,-0.046521,...,-0.008561,-0.008313,-0.011828,-0.008222,NaN,-0.010127,-0.008513,-0.019530,NaN,0.0015
3,2010-03-25,0.017674,-0.013358,NaN,NaN,-0.007997,-0.004150,-0.007961,-0.019029,-0.005630,...,-0.020167,-0.019968,-0.002449,-0.004507,NaN,0.001928,-0.011319,-0.014421,NaN,0.0015
4,2010-03-26,-0.008447,0.017251,NaN,NaN,-0.013084,-0.001101,0.019695,-0.011536,-0.017049,...,0.001670,0.028335,-0.002450,0.002120,NaN,0.004282,-0.004457,0.008490,NaN,0.0015


In [22]:
# Converting excess returns df to long form

# Load the wide format CSV
#excess_returns_df = pd.read_csv('excess_returns_only_wide_format.csv')

# Convert 'Date' to datetime (just in case)
excess_returns_df['Date'] = pd.to_datetime(excess_returns_df['Date'])

# Reshape from wide to long format
# Melt will convert columns like 'AAPL.return' to rows
long_excess_returns = excess_returns_df.melt(
    id_vars=['Date'],
    var_name='Ticker',
    value_name='Excess_Return'
)

# Clean up the 'Ticker' names (remove the '.return' suffix)
long_excess_returns['Ticker'] = long_excess_returns['Ticker'].str.replace('.return', '', regex=False)

# Sort by date and ticker (optional but tidy)
long_excess_returns = long_excess_returns.sort_values(['Date', 'Ticker']).reset_index(drop=True)

# Save to CSV
long_excess_returns.to_csv('excess_returns_long_format.csv', index=False)

print(" Long format file saved as excess_returns_long_format.csv")


 Long format file saved as excess_returns_long_format.csv


In [26]:
long_excess_returns.head(1000)

,Date,Ticker,Excess_Return
0,2010-03-22,A,NaN
1,2010-03-22,AAPL,NaN
2,2010-03-22,ABBV,NaN
3,2010-03-22,ABNB,NaN
4,2010-03-22,ABT,NaN
...,...,...,...
995,2010-03-23,WRB,0.002799
996,2010-03-23,WST,0.012090
997,2010-03-23,WTW,0.008307
998,2010-03-23,WY,0.006507


In [27]:
# CREATING PER-TICKER FILES
# USING EXCESS RETURNS

#Load the long format data
long_excess_returns = pd.read_csv('excess_returns_long_format.csv')

#Create an output folder
output_folder = 'fin_gan_ticker_data'
os.makedirs(output_folder, exist_ok=True)

#Group by ticker and save each as a separate CSV
tickers = long_excess_returns['Ticker'].unique()

for ticker in tickers:
    ticker_df = long_excess_returns[long_excess_returns['Ticker'] == ticker][['Date', 'Excess_Return']]
    
    # Optional: Sort by Date (safety step)
    ticker_df = ticker_df.sort_values('Date').reset_index(drop=True)
    
    # Save the file
    file_path = os.path.join(output_folder, f'{ticker}-data.csv')
    ticker_df.to_csv(file_path, index=False)
    
    print(f"✅ Saved {file_path} with {len(ticker_df)} rows")

print("All per-ticker files generated and ready")


✅ Saved fin_gan_ticker_data/A-data.csv with 3770 rows
✅ Saved fin_gan_ticker_data/AAPL-data.csv with 3770 rows
✅ Saved fin_gan_ticker_data/ABBV-data.csv with 3770 rows
✅ Saved fin_gan_ticker_data/ABNB-data.csv with 3770 rows
✅ Saved fin_gan_ticker_data/ABT-data.csv with 3770 rows
✅ Saved fin_gan_ticker_data/ACGL-data.csv with 3770 rows
✅ Saved fin_gan_ticker_data/ACN-data.csv with 3770 rows
✅ Saved fin_gan_ticker_data/ADBE-data.csv with 3770 rows
✅ Saved fin_gan_ticker_data/ADI-data.csv with 3770 rows
✅ Saved fin_gan_ticker_data/ADM-data.csv with 3770 rows
✅ Saved fin_gan_ticker_data/ADP-data.csv with 3770 rows
✅ Saved fin_gan_ticker_data/ADSK-data.csv with 3770 rows
✅ Saved fin_gan_ticker_data/AEE-data.csv with 3770 rows
✅ Saved fin_gan_ticker_data/AEP-data.csv with 3770 rows
✅ Saved fin_gan_ticker_data/AES-data.csv with 3770 rows
✅ Saved fin_gan_ticker_data/AFL-data.csv with 3770 rows
✅ Saved fin_gan_ticker_data/AIG-data.csv with 3770 rows
✅ Saved fin_gan_ticker_data/AIZ-data.csv wit

In [28]:
# HERE we
# Groups by ticker
#Extract only the Date and Excess_Return columns
#Save each ticker’s data to a Fin-GAN ready CSV file

###  ETF Data preparation

In [29]:
# Computing DAILY RETURNS

# Load ETF data
etf_data = pd.read_csv('etf_data.csv')

# Convert Date to datetime
etf_data['Date'] = pd.to_datetime(etf_data['Date'])
# Sort by Date
etf_data = etf_data.sort_values('Date')

# Compute daily returns for each ETF
for col in etf_data.columns:
    if col.endswith('.close'):
        ticker = col.replace('.close', '')
        return_col = ticker + '.return'
        
        etf_data[return_col] = etf_data[col].pct_change()

# Save the returns-only wide file (optional)
returns_only_cols = ['Date'] + [col for col in etf_data.columns if col.endswith('.return')]
etf_returns_only_df = etf_data[returns_only_cols]

etf_returns_only_df.to_csv('etf_returns_only_wide.csv', index=False)

print(" ETF daily returns computed and saved as etf_returns_only_wide.csv")


 ETF daily returns computed and saved as etf_returns_only_wide.csv


In [30]:
print(etf_returns_only_df.head())

        Date  XLB.return  XLC.return  XLE.return  XLF.return  XLI.return  \
0 2010-03-22         NaN         NaN         NaN         NaN         NaN   
1 2010-03-23    0.013029         NaN    0.003851    0.008243    0.011327   
2 2010-03-24   -0.002924         NaN   -0.005754    0.000629   -0.007040   
3 2010-03-25   -0.019935         NaN   -0.016836    0.004400   -0.000645   
4 2010-03-26    0.007478         NaN    0.000357    0.001252    0.002580   

   XLK.return  XLP.return  XLRE.return  XLU.return  XLV.return  XLY.return  
0         NaN         NaN          NaN         NaN         NaN         NaN  
1    0.009582    0.007883          NaN    0.004032    0.004929    0.003971  
2   -0.004745   -0.007110          NaN   -0.010375   -0.010730   -0.003955  
3    0.001734   -0.001432          NaN   -0.006426   -0.004029    0.005803  
4   -0.004760    0.000717          NaN    0.002383   -0.007467    0.003948  


In [31]:
# COMPUTE EXCESS RETURNS (Subtracting free-risk rate)

# Merge risk-free rate (rf_daily) with ETF returns
etf_excess_returns_df = pd.merge(etf_returns_only_df, rf_daily, left_on='Date', right_index=True, how='left')

# Subtract risk-free rate from each return column
for col in etf_excess_returns_df.columns:
    if col.endswith('.return'):
        etf_excess_returns_df[col] = etf_excess_returns_df[col] - etf_excess_returns_df['risk_free_rate']

# Drop risk-free rate column if not needed
#etf_excess_returns_df = etf_excess_returns_df.drop(columns=['risk_free_rate'])

# Save excess returns wide format
etf_excess_returns_df.to_csv('etf_excess_returns_only_wide.csv', index=False)

print("✅ ETF excess returns wide format file saved as etf_excess_returns_only_wide.csv")


✅ ETF excess returns wide format file saved as etf_excess_returns_only_wide.csv


In [32]:
# Converting from wide to LONG FORMAT

# Melt the wide format to long
long_etf_excess_returns = etf_excess_returns_df.melt(
    id_vars=['Date'],
    var_name='Ticker',
    value_name='Excess_Return'
)

# Clean ticker names
long_etf_excess_returns['Ticker'] = long_etf_excess_returns['Ticker'].str.replace('.return', '', regex=False)

# Sort by Date and Ticker
long_etf_excess_returns = long_etf_excess_returns.sort_values(['Date', 'Ticker']).reset_index(drop=True)

# Save to long format CSV
long_etf_excess_returns.to_csv('etf_excess_returns_long_format.csv', index=False)

print("✅ Long format ETF excess returns saved as etf_excess_returns_long_format.csv")


✅ Long format ETF excess returns saved as etf_excess_returns_long_format.csv


In [33]:
# Generate Per-ETF Files 

# create output folder
etf_output_folder = 'fin_gan_etf_data'
os.makedirs(etf_output_folder, exist_ok=True)

# Get list of ETFs
etf_tickers = long_etf_excess_returns['Ticker'].unique()

#Save per-ETF files
for ticker in etf_tickers:
    etf_ticker_df = long_etf_excess_returns[long_etf_excess_returns['Ticker'] == ticker][['Date', 'Excess_Return']]
    
    # Sort by Date for consistency
    etf_ticker_df = etf_ticker_df.sort_values('Date').reset_index(drop=True)
    
    # Save to CSV
    file_path = os.path.join(etf_output_folder, f'{ticker}-data.csv')
    etf_ticker_df.to_csv(file_path, index=False)
    
    print(f"✅ Saved {file_path} with {len(etf_ticker_df)} rows")

print("All per-ETF files generated")

✅ Saved fin_gan_etf_data/XLB-data.csv with 3770 rows
✅ Saved fin_gan_etf_data/XLC-data.csv with 3770 rows
✅ Saved fin_gan_etf_data/XLE-data.csv with 3770 rows
✅ Saved fin_gan_etf_data/XLF-data.csv with 3770 rows
✅ Saved fin_gan_etf_data/XLI-data.csv with 3770 rows
✅ Saved fin_gan_etf_data/XLK-data.csv with 3770 rows
✅ Saved fin_gan_etf_data/XLP-data.csv with 3770 rows
✅ Saved fin_gan_etf_data/XLRE-data.csv with 3770 rows
✅ Saved fin_gan_etf_data/XLU-data.csv with 3770 rows
✅ Saved fin_gan_etf_data/XLV-data.csv with 3770 rows
✅ Saved fin_gan_etf_data/XLY-data.csv with 3770 rows
✅ Saved fin_gan_etf_data/risk_free_rate-data.csv with 3770 rows
All per-ETF files generated


In [ ]:
# File saving in the dataset folder for replication - STOCK (TICKER)

# # Load the long format excess returns file
# long_excess_returns = pd.read_csv('excess_returns_long_format.csv')

# # Define the correct output path
# output_folder = 'Fin-GAN-main/datasets'  # Already inside Fin-GAN-main/
# assert os.path.exists(output_folder), f"❌ Folder '{output_folder}' not found!"

# # Loop through each ticker and create the expected CSV
# tickers = long_excess_returns['Ticker'].unique()

# for ticker in tickers:
#     df = long_excess_returns[long_excess_returns['Ticker'] == ticker][['Date', 'Excess_Return']].copy()
#     df.dropna(inplace=True)
#     df = df.sort_values('Date').reset_index(drop=True)
    
#     # Rename columns as Fin-GAN expects
#     df.rename(columns={'Date': 'date', 'Excess_Return': 'ret'}, inplace=True)
    
#     # Save to CSV
#     df.to_csv(os.path.join(output_folder, f'{ticker}.csv'), index=False)
#     print(f"✅ Saved {ticker}.csv with {len(df)} rows")

# print("🎉 All per-ticker files successfully created and saved in datasets")



In [ ]:
# ETF

# # Confirm the output folder path
# etf_output_folder = 'Fin-GAN-main/datasets' 

# # Make sure folder exists
# os.makedirs(etf_output_folder, exist_ok=True)

# # Load data
# long_etf_excess_returns = pd.read_csv('etf_excess_returns_long_format.csv')
# # Unique ETF tickers
# etf_tickers = long_etf_excess_returns['Ticker'].unique()

# # Save per-ETF CSVs
# for ticker in etf_tickers:
#     etf_ticker_df = long_etf_excess_returns[long_etf_excess_returns['Ticker'] == ticker][['Excess_Return']]
    
#     # Rename column to "ret" (what FinGAN expects)
#     etf_ticker_df = etf_ticker_df.rename(columns={'Excess_Return': 'ret'}).reset_index(drop=True)
    
#     # Save without date
#     file_path = os.path.join(etf_output_folder, f'{ticker}.csv')
#     etf_ticker_df.to_csv(file_path, index=False)

#     print(f"✅ Saved {file_path} with {len(etf_ticker_df)} rows")

# print("📦 All per-ETF files saved to 'datasets/' folder.")



### ALIGNING WITH FIN GAN

In [10]:
# we rename ret to AdjClose for compatibility with code

# === STOCKS ===
import pandas as pd
import os

# Load your wide-format stock data
stock_data = pd.read_csv('stock_data.csv')
stock_data['Date'] = pd.to_datetime(stock_data['Date'])

# Create output folder for Fin-GAN
output_path = 'Fin-GAN-main/datasets'
os.makedirs(output_path, exist_ok=True)

# Get all unique tickers by splitting column names
tickers = set()
for col in stock_data.columns:
    if '.close' in col:
        tickers.add(col.replace('.close', ''))

# Loop through tickers and extract their data
for ticker in sorted(tickers):
    close_col = f"{ticker}.close"
    open_col = f"{ticker}.open"
    
    if close_col in stock_data.columns and open_col in stock_data.columns:
        df = stock_data[['Date', close_col, open_col]].copy()
        df = df.rename(columns={
            'Date': 'date',
            close_col: 'AdjClose',
            open_col: 'AdjOpen'
        })
        df = df.dropna().sort_values('date').reset_index(drop=True)

        # Save to CSV
        file_path = os.path.join(output_path, f"{ticker}.csv")
        df.to_csv(file_path, index=False)
        print(f"✅ Saved {ticker}.csv with {len(df)} rows")

print("🎯 All stock CSVs are now Fin-GAN-ready!")


/var/folders/k1/p_xw0_4x4zd812685rsx20380000gn/T/ipykernel_36895/1694104234.py:9: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  stock_data['Date'] = pd.to_datetime(stock_data['Date'])


✅ Saved A.csv with 3770 rows
✅ Saved AAPL.csv with 3770 rows
✅ Saved ABBV.csv with 3069 rows
✅ Saved ABNB.csv with 1069 rows
✅ Saved ABT.csv with 3770 rows
✅ Saved ACGL.csv with 3770 rows
✅ Saved ACN.csv with 3770 rows
✅ Saved ADBE.csv with 3770 rows
✅ Saved ADI.csv with 3770 rows
✅ Saved ADM.csv with 3770 rows
✅ Saved ADP.csv with 3770 rows
✅ Saved ADSK.csv with 3770 rows
✅ Saved AEE.csv with 3770 rows
✅ Saved AEP.csv with 3770 rows
✅ Saved AES.csv with 3770 rows
✅ Saved AFL.csv with 3770 rows
✅ Saved AIG.csv with 3770 rows
✅ Saved AIZ.csv with 3770 rows
✅ Saved AJG.csv with 3770 rows
✅ Saved AKAM.csv with 3770 rows
✅ Saved ALB.csv with 3770 rows
✅ Saved ALGN.csv with 3770 rows
✅ Saved ALL.csv with 3770 rows
✅ Saved ALLE.csv with 2847 rows
✅ Saved AMAT.csv with 3770 rows
✅ Saved AMCR.csv with 3227 rows
✅ Saved AMD.csv with 3770 rows
✅ Saved AME.csv with 3770 rows
✅ Saved AMGN.csv with 3770 rows
✅ Saved AMP.csv with 3770 rows
✅ Saved AMT.csv with 3770 rows
✅ Saved AMZN.csv with 3770 ro

In [11]:
# === ETFS ===

# Load the ETF-wide CSV
etf_data = pd.read_csv('etf_data.csv')
etf_data['Date'] = pd.to_datetime(etf_data['Date'])

# Output folder
output_folder = 'Fin-GAN-main/datasets'
os.makedirs(output_folder, exist_ok=True)

# Extract all unique ETF tickers from column names
etf_tickers = set()
for col in etf_data.columns:
    if col.endswith('.close'):
        etf_tickers.add(col.replace('.close', ''))

print(f"📈 Found {len(etf_tickers)} ETF tickers")

# Generate per-ETF files
for ticker in sorted(etf_tickers):
    close_col = f'{ticker}.close'
    open_col = f'{ticker}.open'
    
    if close_col in etf_data.columns and open_col in etf_data.columns:
        df = pd.DataFrame({
            'date': etf_data['Date'],
            'AdjClose': etf_data[close_col],
            'AdjOpen': etf_data[open_col]
        })

        df.dropna(inplace=True)
        df = df.sort_values('date').reset_index(drop=True)

        filename = os.path.join(output_folder, f'{ticker}.csv')
        df.to_csv(filename, index=False)
        print(f"✅ Saved {filename} with {len(df)} rows")
    else:
        print(f"⚠️ Missing data for {ticker}, skipping...")

print("🎯 All ETF files saved in Fin-GAN-main/datasets")


📈 Found 11 ETF tickers
✅ Saved Fin-GAN-main/datasets/XLB.csv with 3770 rows
✅ Saved Fin-GAN-main/datasets/XLC.csv with 1694 rows
✅ Saved Fin-GAN-main/datasets/XLE.csv with 3770 rows
✅ Saved Fin-GAN-main/datasets/XLF.csv with 3770 rows
✅ Saved Fin-GAN-main/datasets/XLI.csv with 3770 rows
✅ Saved Fin-GAN-main/datasets/XLK.csv with 3770 rows
✅ Saved Fin-GAN-main/datasets/XLP.csv with 3770 rows
✅ Saved Fin-GAN-main/datasets/XLRE.csv with 2372 rows
✅ Saved Fin-GAN-main/datasets/XLU.csv with 3770 rows
✅ Saved Fin-GAN-main/datasets/XLV.csv with 3770 rows
✅ Saved Fin-GAN-main/datasets/XLY.csv with 3770 rows
🎯 All ETF files saved in Fin-GAN-main/datasets


/var/folders/k1/p_xw0_4x4zd812685rsx20380000gn/T/ipykernel_36895/1500372166.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  etf_data['Date'] = pd.to_datetime(etf_data['Date'])


In [1]:
# ALIGNED FILES TO AVOID MISMATCH WITH NAN

import pandas as pd
import os

# === CONFIG ===
stock_data_file = 'stock_data.csv'
etf_data_file = 'etf_data.csv'
mapping_file = 'Fin-GAN-main/datasets/stocks-etfs-list.csv'
output_path = 'Fin-GAN-main/datasets'

# === SETUP ===
os.makedirs(output_path, exist_ok=True)

stock_data = pd.read_csv(stock_data_file)
etf_data = pd.read_csv(etf_data_file)
mapping_df = pd.read_csv(mapping_file)

stock_data['Date'] = pd.to_datetime(stock_data['Date'])
etf_data['Date'] = pd.to_datetime(etf_data['Date'])

stock_data.set_index('Date', inplace=False)
etf_data.set_index('Date', inplace=False)

# === MAIN LOOP: Process each stock and its ETF ===
for _, row in mapping_df.iterrows():
    stock = row['ticker_x']
    etf = row['SectorTicker']

    stock_close = f"{stock}.close"
    stock_open = f"{stock}.open"
    etf_close = f"{etf}.close"
    etf_open = f"{etf}.open"

    # Check required columns
    if all(col in stock_data.columns for col in [stock_close, stock_open]) and \
       all(col in etf_data.columns for col in [etf_close, etf_open]):

        # Extract data
        s_df = stock_data[['Date', stock_close, stock_open]].copy()
        s_df.columns = ['date', 'AdjClose_stock', 'AdjOpen_stock']

        e_df = etf_data[['Date', etf_close, etf_open]].copy()
        e_df.columns = ['date', 'AdjClose_etf', 'AdjOpen_etf']

        # Merge and align
        merged = pd.merge(s_df, e_df, on='date', how='inner')
        merged.dropna(inplace=True)

        # === Save aligned stock ===
        s_out = merged[['date', 'AdjClose_stock', 'AdjOpen_stock']].rename(
            columns={'AdjClose_stock': 'AdjClose', 'AdjOpen_stock': 'AdjOpen'}
        )
        s_out.to_csv(os.path.join(output_path, f"{stock}.csv"), index=False)

        # === Save aligned ETF ===
        e_out = merged[['date', 'AdjClose_etf', 'AdjOpen_etf']].rename(
            columns={'AdjClose_etf': 'AdjClose', 'AdjOpen_etf': 'AdjOpen'}
        )
        e_out.to_csv(os.path.join(output_path, f"{etf}.csv"), index=False)

        print(f"✅ Saved aligned: {stock}.csv and {etf}.csv")

    else:
        print(f"⚠️ Skipped {stock} or {etf} — missing columns")

print("🎯 All files are clean, aligned, and FinGAN-ready!")


/var/folders/k1/p_xw0_4x4zd812685rsx20380000gn/T/ipykernel_39775/1034287269.py:19: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  stock_data['Date'] = pd.to_datetime(stock_data['Date'])
/var/folders/k1/p_xw0_4x4zd812685rsx20380000gn/T/ipykernel_39775/1034287269.py:20: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  etf_data['Date'] = pd.to_datetime(etf_data['Date'])


✅ Saved aligned: MMM.csv and XLI.csv
✅ Saved aligned: AOS.csv and XLI.csv
✅ Saved aligned: ABT.csv and XLV.csv
✅ Saved aligned: ABBV.csv and XLV.csv
✅ Saved aligned: ACN.csv and XLK.csv
✅ Saved aligned: ADBE.csv and XLK.csv
✅ Saved aligned: AMD.csv and XLK.csv
✅ Saved aligned: AES.csv and XLU.csv
✅ Saved aligned: AFL.csv and XLF.csv
✅ Saved aligned: A.csv and XLV.csv
✅ Saved aligned: APD.csv and XLB.csv
✅ Saved aligned: ABNB.csv and XLY.csv
✅ Saved aligned: AKAM.csv and XLK.csv
✅ Saved aligned: ALB.csv and XLB.csv
✅ Saved aligned: ARE.csv and XLRE.csv
✅ Saved aligned: ALGN.csv and XLV.csv
✅ Saved aligned: ALLE.csv and XLI.csv
✅ Saved aligned: LNT.csv and XLU.csv
✅ Saved aligned: ALL.csv and XLF.csv
✅ Saved aligned: GOOGL.csv and XLC.csv
✅ Saved aligned: GOOG.csv and XLC.csv
✅ Saved aligned: MO.csv and XLP.csv
✅ Saved aligned: AMZN.csv and XLY.csv
✅ Saved aligned: AMCR.csv and XLB.csv
✅ Saved aligned: AEE.csv and XLU.csv
✅ Saved aligned: AEP.csv and XLU.csv
✅ Saved aligned: AXP.csv and 